In [1]:
import pandas as pd
import numpy as np


In [2]:
df = pd.read_csv("../data/raw/big_startup_success_dataset.csv")

In [3]:
df.shape

(66368, 14)

In [4]:
df.head(10)

,permalink,name,homepage_url,category_list,funding_total_usd,status,country_code,state_code,region,city,funding_rounds,founded_at,first_funding_at,last_funding_at
0,/organization/-fame,#fame,http://livfame.com,Media,10000000,operating,IND,16,Mumbai,Mumbai,1,NaN,2015-01-05,2015-01-05
1,/organization/-qounter,:Qounter,http://www.qounter.com,Application Platforms|Real Time|Social Network...,700000,operating,USA,DE,DE - Other,Delaware City,2,2014-09-04,2014-03-01,2014-10-14
2,/organization/-the-one-of-them-inc-,"(THE) ONE of THEM,Inc.",http://oneofthem.jp,Apps|Games|Mobile,3406878,operating,NaN,NaN,NaN,NaN,1,NaN,2014-01-30,2014-01-30
3,/organization/0-6-com,0-6.com,http://www.0-6.com,Curated Web,2000000,operating,CHN,22,Beijing,Beijing,1,2007-01-01,2008-03-19,2008-03-19
4,/organization/004-technologies,004 Technologies,http://004gmbh.de/en/004-interact,Software,-,operating,USA,IL,"Springfield, Illinois",Champaign,1,2010-01-01,2014-07-24,2014-07-24
5,/organization/01games-technology,01Games Technology,http://www.01games.hk/,Games,41250,operating,HKG,NaN,Hong Kong,Hong Kong,1,NaN,2014-07-01,2014-07-01
6,/organization/0ndine-biomedical-inc,Ondine Biomedical Inc.,http://ondinebio.com,Biotechnology,762851,operating,CAN,BC,Vancouver,Vancouver,2,1997-01-01,2009-09-11,2009-12-21
7,/organization/0xdata,H2O.ai,http://h2o.ai/,Analytics,33600000,operating,USA,CA,SF Bay Area,Mountain View,4,2011-01-01,2013-01-03,2015-11-09
8,/organization/1,One Inc.,http://whatis1.com,Mobile,1150050,operating,USA,CA,SF Bay Area,San Francisco,3,2011-08-01,2011-07-20,2014-02-05
9,/organization/1-2-3-listo,"1,2,3 Listo",http://www.123listo.com,E-Commerce,40000,operating,CHL,12,Santiago,Las Condes,1,2012-01-01,2013-02-18,2013-02-18


In [5]:
df.isnull().sum()

permalink                0
name                     1
homepage_url          5058
category_list         3148
funding_total_usd        0
status                   0
country_code          6958
state_code            8547
region                8030
city                  8028
funding_rounds           0
founded_at           15221
first_funding_at        24
last_funding_at          0
dtype: int64

In [6]:
df = df.drop(columns=['homepage_url', 'permalink'])

In [7]:
df = df.dropna(subset=['name', 'first_funding_at'])

In [8]:
# ── STEP 3: Fix dates FIRST (before any filling) ──
for col in ['founded_at', 'first_funding_at', 'last_funding_at']:
    df[col] = pd.to_datetime(df[col], errors='coerce', format='mixed')

# Flag BEFORE filling — True = has a real founded date
df['has_founded_date'] = df['founded_at'].notna()


In [9]:
df[["country_code", "state_code", "region", "city", "category_list"]] = (
    df[["country_code", "state_code", "region", "city", "category_list"]]
    .fillna("Unknown")
)

In [10]:
df['funding_total_usd']=df['funding_total_usd'].replace('-', pd.NA)

In [11]:
df.isnull().sum()

name                     0
category_list            0
funding_total_usd    12784
status                   0
country_code             0
state_code               0
region                   0
city                     0
funding_rounds           0
founded_at           15218
first_funding_at         0
last_funding_at          0
has_founded_date         0
dtype: int64

In [12]:
df.dtypes

name                            str
category_list                   str
funding_total_usd               str
status                          str
country_code                    str
state_code                      str
region                          str
city                            str
funding_rounds                int64
founded_at           datetime64[us]
first_funding_at     datetime64[us]
last_funding_at      datetime64[us]
has_founded_date               bool
dtype: object

In [13]:
df.head()

,name,category_list,funding_total_usd,status,country_code,state_code,region,city,funding_rounds,founded_at,first_funding_at,last_funding_at,has_founded_date
0,#fame,Media,10000000,operating,IND,16,Mumbai,Mumbai,1,NaT,2015-01-05,2015-01-05,False
1,:Qounter,Application Platforms|Real Time|Social Network...,700000,operating,USA,DE,DE - Other,Delaware City,2,2014-09-04,2014-03-01,2014-10-14,True
2,"(THE) ONE of THEM,Inc.",Apps|Games|Mobile,3406878,operating,Unknown,Unknown,Unknown,Unknown,1,NaT,2014-01-30,2014-01-30,False
3,0-6.com,Curated Web,2000000,operating,CHN,22,Beijing,Beijing,1,2007-01-01,2008-03-19,2008-03-19,True
4,004 Technologies,Software,NaN,operating,USA,IL,"Springfield, Illinois",Champaign,1,2010-01-01,2014-07-24,2014-07-24,True


In [14]:
df["funding_total_usd"] = df["funding_total_usd"].fillna(0)

In [15]:
df["funding_total_usd"] = pd.to_numeric(
    df["funding_total_usd"],
    errors="coerce"
)

In [16]:

df["funding_total_usd"].dtype


dtype('float64')

In [17]:
for col in ['founded_at', 'first_funding_at', 'last_funding_at']:
     df[col] = pd.to_datetime(df[col], errors='coerce')

In [18]:
df.dtypes

name                            str
category_list                   str
funding_total_usd           float64
status                          str
country_code                    str
state_code                      str
region                          str
city                            str
funding_rounds                int64
founded_at           datetime64[us]
first_funding_at     datetime64[us]
last_funding_at      datetime64[us]
has_founded_date               bool
dtype: object

In [19]:
df['status'] = df['status'].str.strip().str.lower()
df['country_code'] = df['country_code'].str.strip().str.upper()

In [20]:
df['primary_category'] = (
    df['category_list']
    .str.split('|')
    .str[0]
    .str.strip()
    .str.title()
)

In [21]:
print("Types fixed. Sample:")
df[['funding_total_usd','status','primary_category']].head(3)

Types fixed. Sample:


,funding_total_usd,status,primary_category
0,10000000.0,operating,Media
1,700000.0,operating,Application Platforms
2,3406878.0,operating,Apps


In [22]:
# Create binary target column for failed startups
df['is_failed'] = (df['status'] == 'closed').astype(int)

In [23]:
# Survival time in days(founded_at to last_funding_at)
df['survival_days'] = (df['last_funding_at'] - df['founded_at']).dt.days

In [24]:
df['survival_years'] = (df['survival_days']/365).round(1)

In [25]:
#Funding size buckets
df['funding_bucket'] = pd.cut(
    df['funding_total_usd'],
    bins=[0,100000, 1000000, 10000000, float('inf')],
    labels=['Micro', 'Small', 'Medium', 'Large']
)

In [26]:
# Founded year
df['founded_year'] = df['founded_at'].dt.year

# Era classification
df['era'] = pd.cut(
    df['founded_year'],
    bins=[1999, 2009, 2014, 2020, 2025],
    labels=['2000-2009', '2010-2014', '2015-2020', '2021-2025']
)

In [27]:
df[['is_failed','survival_years','funding_bucket','founded_year','era']].head()

,is_failed,survival_years,funding_bucket,founded_year,era
0,0,NaN,Medium,NaN,NaN
1,0,0.1,Small,2014.0,2010-2014
2,0,NaN,Medium,NaN,NaN
3,0,1.2,Medium,2007.0,2000-2009
4,0,4.6,NaN,2010.0,2010-2014


In [28]:
#Final validation
print("=== CLEAN DATA SUMMARY ===")

print(f"Shape: {df.shape}")
print(f"Remaining Nulls: {df.isnull().sum().sum()}")
print(f"Failure Rate: {df['is_failed'].mean().round(3)}")

=== CLEAN DATA SUMMARY ===
Shape: (66343, 20)
Remaining Nulls: 93653
Failure Rate: 0.094


In [29]:
# Save
df.to_csv('../data/cleaned/startups_clean.csv', index=False)
print("Saved!")

Saved!
